In [1]:
# 01_eda.ipynb
# =============================================================================
# Exploratory analysis of the heterogeneous SMEsD graph: node/edge counts,
# per-relation edge-count distribution (dense economic ties 0-11 vs sparse
# financial links 12-19), label balance across the strict (mask A) splits, and
# a visual summary of the leakage control. Figures are grayscale, dpi 600,
# saved as png and pdf, with no embedded captions.
# =============================================================================
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[0]
PROCESSED = ROOT / "data" / "processed"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for p in (FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "savefig.bbox": "tight",
                     "font.size": 10, "axes.grid": True})
sns.set_theme(style="whitegrid", palette="Greys_r")
GREY_D, GREY_L = "#333333", "#999999"


def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)


# --- Load graph --------------------------------------------------------------
data = torch.load(PROCESSED / "smesd_hetero.pt", weights_only=False)
comp = data["company"]
N_COMP = comp.num_nodes
N_PERS = data["person"].num_nodes

# --- 1. Per-relation edge counts (group typed edges by relation integer) ------
rel_counts = defaultdict(int)
for et in data.edge_types:
    rel = int(et[1].split("_")[1])
    rel_counts[rel] += data[et].edge_index.size(1)
rel_df = pd.DataFrame({"relation": list(range(20)),
                       "edges": [rel_counts.get(r, 0) for r in range(20)]})
rel_df["group"] = np.where(rel_df["relation"] <= 11, "economic (0-11)", "financial (12-19)")
rel_df.to_csv(TAB / "01_edge_counts.csv", index=False)
print("edge counts per relation:")
print(rel_df.to_string(index=False))
print(f"\ntotal edges={int(rel_df['edges'].sum()):,}  "
      f"economic(0-11)={int(rel_df[rel_df.relation<=11]['edges'].sum()):,}  "
      f"financial(12-19)={int(rel_df[rel_df.relation>=12]['edges'].sum()):,}")

# --- Figure: edges per relation, log scale, grayscale by group ---------------
fig, ax = plt.subplots(figsize=(7, 3.2))
colors = [GREY_D if g == "economic (0-11)" else GREY_L for g in rel_df["group"]]
ax.bar(rel_df["relation"], rel_df["edges"].clip(lower=0.5), color=colors,
       edgecolor="black", linewidth=0.5)
ax.axhline(15, color="black", linestyle="--", linewidth=0.8)
ax.set_yscale("log")
ax.set_xlabel("Relation type")
ax.set_ylabel("Number of edges (log scale)")
ax.set_xticks(range(20))
handles = [plt.Rectangle((0, 0), 1, 1, color=GREY_D, ec="black"),
           plt.Rectangle((0, 0), 1, 1, color=GREY_L, ec="black")]
ax.legend(handles, ["Economic ties (0-11)", "Financial links (12-19)"],
          frameon=True, fontsize=8)
savefig(fig, "fig_edge_counts")

# --- 2. Label balance across strict (mask A) splits --------------------------
y = comp.y.numpy()
splits = {"Train": comp.train_mask.numpy(), "Validation": comp.val_mask.numpy(),
          "Test": comp.test_mask.numpy()}
bal_rows = []
for nm, m in splits.items():
    yy = y[m]
    bal_rows.append({"split": nm, "n": int(m.sum()), "positives": int(yy.sum()),
                     "positive_rate": round(float(yy.mean()), 4)})
bal_df = pd.DataFrame(bal_rows)
bal_df.to_csv(TAB / "01_label_balance.csv", index=False)
print("\nlabel balance (mask A, strict):")
print(bal_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(4.5, 3.2))
xpos = np.arange(len(bal_df))
pos = bal_df["positives"].values
neg = bal_df["n"].values - pos
ax.bar(xpos, neg, color=GREY_L, edgecolor="black", linewidth=0.5, label="Surviving")
ax.bar(xpos, pos, bottom=neg, color=GREY_D, edgecolor="black", linewidth=0.5,
       label="Bankrupt")
ax.set_xticks(xpos); ax.set_xticklabels(bal_df["split"])
ax.set_ylabel("Number of companies")
ax.legend(frameon=True, fontsize=8)
savefig(fig, "fig_label_balance")

# --- 3. Leakage control summary ----------------------------------------------
idmaps = torch.load(PROCESSED / "smesd_idmaps.pt", weights_only=False)
n_dup = int(len(idmaps["dup_ids"]))
a = {k: int(comp[f"{k}_mask"].sum()) for k in ["train", "val", "test"]}
b = {k: int(comp[f"{k}_mask_repro"].sum()) for k in ["train", "val", "test"]}
leak_df = pd.DataFrame([
    {"mask": "A (strict)", **a, "duplicated_removed": n_dup},
    {"mask": "B (as-shipped)", **b, "duplicated_removed": 0}])
leak_df.to_csv(TAB / "01_leakage_summary.csv", index=False)
print(f"\nleakage control: {n_dup} duplicated companies; mask A removes them from "
      f"earlier splits (disjoint), mask B keeps them.")
print(leak_df.to_string(index=False))

# --- Figure: mask A vs B split sizes -----------------------------------------
fig, ax = plt.subplots(figsize=(5.2, 3.2))
labels = ["Train", "Validation", "Test"]
xpos = np.arange(len(labels)); width = 0.38
ax.bar(xpos - width / 2, [a["train"], a["val"], a["test"]], width,
       color=GREY_D, edgecolor="black", linewidth=0.5, label="Mask A (strict)")
ax.bar(xpos + width / 2, [b["train"], b["val"], b["test"]], width,
       color=GREY_L, edgecolor="black", linewidth=0.5, label="Mask B (as-shipped)")
ax.set_xticks(xpos); ax.set_xticklabels(labels)
ax.set_ylabel("Number of companies")
ax.legend(frameon=True, fontsize=8)
savefig(fig, "fig_leakage_masks")

print("\n01_eda complete: 3 figures (edge counts, label balance, leakage masks) "
      "and 3 tables saved.")


edge counts per relation:
 relation  edges             group
        0    698   economic (0-11)
        1    698   economic (0-11)
        2    883   economic (0-11)
        3    883   economic (0-11)
        4     37   economic (0-11)
        5     37   economic (0-11)
        6    938   economic (0-11)
        7    942   economic (0-11)
        8   3657   economic (0-11)
        9   3657   economic (0-11)
       10    376   economic (0-11)
       11    376   economic (0-11)
       12     53 financial (12-19)
       13      9 financial (12-19)
       14      2 financial (12-19)
       15     13 financial (12-19)
       16     53 financial (12-19)
       17      9 financial (12-19)
       18      2 financial (12-19)
       19     13 financial (12-19)

total edges=13,336  economic(0-11)=13,182  financial(12-19)=154



label balance (mask A, strict):
     split    n  positives  positive_rate
     Train 2765       1590         0.5750
Validation  720        354         0.4917
      Test  491        318         0.6477



leakage control: 51 duplicated companies; mask A removes them from earlier splits (disjoint), mask B keeps them.
          mask  train  val  test  duplicated_removed
    A (strict)   2765  720   491                  51
B (as-shipped)   2816  721   491                   0



01_eda complete: 3 figures (edge counts, label balance, leakage masks) and 3 tables saved.
